## Kernel to load: vax_inc_general 

In [3]:
import pandas as pd
import numpy as np
import pycountry
import sys,os,json
import glob
import re
from pathlib import Path


In [4]:
notebook_dir = os.path.dirname(os.getcwd())
source_data_path=os.path.join(notebook_dir, "Common Source Data")
sys.path.append(source_data_path)
from country_codes import countries

In [5]:
imputed_df=pd.read_csv("FINAL_ALL_incidence_imputations_to_add.csv")
imputed_df

,ISO3,Year,Animal,Disease,"Incidence (Cases per 100,000)",Source,Incidence Lower (Residual Only),Incidence Upper (Residual Only),Incidence Lower (Bootstrap Only),Incidence Upper (Bootstrap Only),Incidence Lower (Asymmetric),Incidence Upper (Asymmetric)
0,ABW,2005.0,Swine,African swine fever virus (Inf. with),2055.570312,Imputed,1976.645986,2154.658744,1531.653900,2497.063500,1452.729604,2596.151908
1,ABW,2005.0,Cattle,Anthrax,2.397447,Imputed,0.639392,5.121005,0.161589,5.864545,0.000000,8.588103
2,ABW,2005.0,Swine,Anthrax,7.689636,Imputed,4.468805,12.157573,4.283852,11.751440,1.063021,16.219376
3,ABW,2005.0,Swine,Aujeszky's disease virus (Inf. with),11.338800,Imputed,9.622134,13.819834,5.851690,16.285545,4.135023,18.766579
4,ABW,2005.0,Poultry,Avian infectious bronchitis,31.666777,Imputed,21.728277,42.612208,8.873716,78.729260,0.000000,89.674694
...,...,...,...,...,...,...,...,...,...,...,...,...
175353,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
175354,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
175355,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
175356,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
imputed_df=pd.read_csv("FINAL_ALL_incidence_imputations_to_add.csv")
imputed_df['Source']='Imputed'
imputed_df.drop(columns=['Incidence Lower (Residual Only)','Incidence Upper (Residual Only)',
                        'Incidence Lower (Bootstrap Only)','Incidence Upper (Bootstrap Only)'],inplace=True)
imputed_df.rename(columns={'Incidence Lower (Asymmetric)':'Incidence (Cases per 100,000) Lower',
                           'Incidence Upper (Asymmetric)':'Incidence (Cases per 100,000) Upper'},inplace=True)



poultry_original=pd.read_csv("2005-2025_poultry_disease_incidence_stats.csv")
poultry_original['Animal']='Poultry'

cattle_original=pd.read_csv("2005-2025_cattle_disease_incidence_stats.csv")
cattle_original['Animal']='Cattle'

swine_original=pd.read_csv("2005-2025_swine_disease_incidence_stats.csv")
swine_original['Animal']='Swine'

orginal_dfs=pd.concat([poultry_original, cattle_original, swine_original])
orginal_dfs.rename(columns={'Incidence':'Incidence (Cases per 100,000)',
                            'Incidence Lower':'Incidence (Cases per 100,000) Lower',
                           'Incidence Upper':'Incidence (Cases per 100,000) Upper'},inplace=True)


orginal_dfs['Incidence (Cases per 100,000)']*=100000
orginal_dfs['Incidence (Cases per 100,000) Lower']*=100000
orginal_dfs['Incidence (Cases per 100,000) Upper']*=100000


orginal_dfs['ISO3']=[countries[country] for country in orginal_dfs['Country']]
#orginal_dfs['Source']='WAHIS administrative division reports'
orginal_dfs['Source'] = orginal_dfs['Derived_Cases_Method'].apply(
        lambda x: 'WAHIS administrative division reports' if x == 'None' else 'WAHIS administrative division reports (includes linear interpolation)'
    )

orginal_dfs.drop(columns=['Derived_Cases_Method'],inplace=True)
compiled=pd.concat([orginal_dfs,imputed_df])



In [4]:
countries['Serbia and Montenegro']='SCG'

flipped_dict = {v: k for k, v in countries.items()}
flipped_dict['TUR']='Türkiye, Republic of'
flipped_dict['XKX']='Kosovo'

In [5]:
country = compiled['ISO3'].map(flipped_dict)
compiled = (compiled
    .assign(Country=country)
    .loc[~((country.eq('Serbia and Montenegro')) & (compiled['Year'] > 2006))].copy()
    .rename(columns={'Year Range Incidence Estimate': 'Year Cases Data'})
)


In [6]:
compiled['Animal']=['Pigs' if i=='Swine' else i for i in compiled['Animal']]

In [7]:
compiled=compiled[['Country','ISO3','Animal','Year','Disease','Incidence (Cases per 100,000)','Source','Incidence (Cases per 100,000) Lower', 'Incidence (Cases per 100,000) Upper', 'Latest Reported Cases Aggregate','Cases','Cases Lower',
                         'Cases Upper','Year Cases Data']]

In [8]:
compiled.drop(columns=['Cases','Cases Upper','Cases Lower'],inplace=True)

In [9]:
vaccine_df=pd.read_csv(os.path.join(notebook_dir,'Vaccination Coverage','Supplementary Spreadsheet- Vaccination Coverage Estimates.csv'))

In [10]:
df_pop_cattle=pd.read_csv(os.path.join(source_data_path, 'Processed data','FAO Populations','cattle_pop.csv'))
df_pop_cattle = df_pop_cattle.sort_values('Value').drop_duplicates(subset=['ISO3','Year','Item'], keep='last').loc[:,['Area','Value','ISO3','Year']]
df_pop_cattle=df_pop_cattle.sort_values('Year').loc[:,['Area','Value','ISO3','Year']]
df_pop_cattle['Animal']=['Cattle']*df_pop_cattle.shape[0]
df_pop_cattle.rename(columns={'Value':'latest pop'},inplace=True)


df_pop_poultry=pd.read_csv(os.path.join(source_data_path,'Processed data','FAO Populations', "poultry_pop.csv"))
df_pop_poultry = df_pop_poultry.sort_values('Value').drop_duplicates(subset=['ISO3','Year','Item'], keep='last')
df_pop_poultry = (
    df_pop_poultry.groupby(['ISO3', 'Year Code'], as_index=False)
    .agg({
        'Domain Code': 'first',
        'Domain': 'first',
        'Area Code (M49)': 'first',
        'Area': 'first',
        'Element Code': 'first',
        'Element': 'first',
        'Item Code (CPC)': 'first',
        'Year Code': 'first',
        'Year': 'first',
        'Unit': 'first',
        'Value': 'sum', 
        'Flag': 'first',
        'Flag Description': 'first',
        'Note': 'first',
        'ISO3':'first'
    })
)
df_pop_poultry['Item'] = 'Poultry'
df_pop_poultry=df_pop_poultry.sort_values('Year').loc[:,['Area','Value','ISO3','Year']]
df_pop_poultry.rename(columns={'Value':'latest pop'},inplace=True)
df_pop_poultry['Animal']=['Poultry']*df_pop_poultry.shape[0]
df_pop_poultry['latest pop']*=1000

df_pop_swine=pd.read_csv(os.path.join(source_data_path,'Processed data','FAO Populations', 'swine_pop.csv'))
df_pop_swine = df_pop_swine.sort_values('Value').drop_duplicates(subset=['ISO3','Year','Item'], keep='last')
df_pop_swine=df_pop_swine.sort_values('Year').loc[:,['Area','Value','ISO3','Year']]
df_pop_swine.rename(columns={'Value':'latest pop'},inplace=True)
df_pop_swine['Animal']=['Pigs']*df_pop_swine.shape[0]

df_pop_killed_cattle=pd.read_csv(os.path.join(source_data_path,'Processed data','FAO Populations', 'killed_cattle_pop.csv'))
df_pop_killed_cattle = df_pop_killed_cattle.sort_values('Value').drop_duplicates(subset=['ISO3','Year','Item'], keep='last')
df_pop_killed_cattle.rename(columns={'Value':'latest pop'},inplace=True)
df_pop_killed_cattle=df_pop_killed_cattle.sort_values('Year').loc[:,['Area','latest pop','ISO3','Year']]
df_pop_killed_cattle['Animal']=['Cattle']*df_pop_killed_cattle.shape[0]

df_pop_killed_poultry=pd.read_csv(os.path.join(source_data_path, 'Processed data','FAO Populations', "killed_poultry_pop.csv"))
df_pop_killed_poultry = df_pop_killed_poultry.sort_values('Value').drop_duplicates(subset=['ISO3','Year','Item'], keep='last')
df_pop_killed_poultry = (
    df_pop_killed_poultry.groupby(['ISO3', 'Year Code'], as_index=False)
    .agg({
        'Domain Code': 'first',
        'Domain': 'first',
        'Area Code (M49)': 'first',
        'Area': 'first',
        'Element Code': 'first',
        'Element': 'first',
        'Item Code (CPC)': 'first',
        'Year Code': 'first',
        'Year': 'first',
        'Unit': 'first',
        'Value': 'sum',  
        'Flag': 'first',
        'Flag Description': 'first',
        'Note': 'first',
        'ISO3':'first'
    })
)
df_pop_killed_poultry['Item'] = 'Poultry'
df_pop_killed_poultry=df_pop_killed_poultry.sort_values('Year').loc[:,['Area','Value','ISO3','Year']]
df_pop_killed_poultry.rename(columns={'Value':'latest pop'},inplace=True)
df_pop_killed_poultry['Animal']=['Poultry']*df_pop_killed_poultry.shape[0]
df_pop_killed_poultry['latest pop']*=1000

df_pop_killed_swine=pd.read_csv(os.path.join(source_data_path,'Processed data','FAO Populations', 'killed_swine_pop.csv'))
df_pop_killed_swine = df_pop_killed_swine.sort_values('Value').drop_duplicates(subset=['ISO3','Year','Item'], keep='last')
df_pop_killed_swine=df_pop_killed_swine.sort_values('Year').loc[:,['Area','Value','ISO3','Year']]
df_pop_killed_swine.rename(columns={'Value':'latest pop'},inplace=True)
df_pop_killed_swine['Animal']=['Pigs']*df_pop_killed_swine.shape[0]

In [11]:
df_pops=pd.concat([df_pop_cattle,df_pop_poultry,df_pop_swine]).drop(columns=['Area'])

In [12]:
df_pops_killed=pd.concat([df_pop_killed_cattle,df_pop_killed_poultry,df_pop_killed_swine]).drop(columns=['Area'])

In [13]:
df_pops_killed.rename(columns={'latest pop':'latest killed pop'},inplace=True)

In [14]:
compiled=compiled.merge(df_pops,how='left',on=['ISO3','Animal','Year'])
compiled=compiled.merge(df_pops_killed,how='left',on=['ISO3','Animal','Year'])


In [15]:
compiled['Estimated (Scaled) Cases'] = pd.to_numeric((
    compiled['Incidence (Cases per 100,000)'] *
    (compiled['latest pop'] + compiled['latest killed pop']) / 100000
).round(), errors='coerce')


In [16]:
compiled.rename(columns={'Latest Reported Cases Aggregate':'Aggregate of Latest Reported Cases (up to Year)',
                        'Year Cases Data':'Year Data'},inplace=True)
compiled.drop(columns=['latest pop','latest killed pop'],inplace=True)
# Reorder last two columns
columns = compiled.columns.tolist()
columns[-2], columns[-1] = columns[-1], columns[-2]  # Swap the last two columns
compiled = compiled[columns]
compiled

,Country,ISO3,Animal,Year,Disease,"Incidence (Cases per 100,000)",Source,"Incidence (Cases per 100,000) Lower","Incidence (Cases per 100,000) Upper",Aggregate of Latest Reported Cases (up to Year),Estimated (Scaled) Cases,Year Data
0,Afghanistan,AFG,Poultry,2005.0,Newcastle disease (velogenic),1.703944,WAHIS administrative division reports,1.392092,2.071751,467.0,934.0,2005
1,Angola,AGO,Poultry,2005.0,Newcastle disease (velogenic),1.187285,WAHIS administrative division reports,0.962039,1.445195,93.0,186.0,2005
2,Albania,ALB,Poultry,2005.0,Newcastle disease (velogenic),3.037240,WAHIS administrative division reports,2.677506,3.427814,252.0,504.0,2005
3,Austria,AUT,Poultry,2005.0,Avian chlamydiosis,0.005042,WAHIS administrative division reports,0.000838,0.015558,4.0,4.0,2005
4,Burundi,BDI,Poultry,2005.0,Fowl typhoid,37.293640,WAHIS administrative division reports,34.577796,40.147984,1378.0,1378.0,2005
...,...,...,...,...,...,...,...,...,...,...,...,...
232709,NaN,NaN,NaN,NaN,NaN,NaN,Imputed,NaN,NaN,NaN,NaN,NaN
232710,NaN,NaN,NaN,NaN,NaN,NaN,Imputed,NaN,NaN,NaN,NaN,NaN
232711,NaN,NaN,NaN,NaN,NaN,NaN,Imputed,NaN,NaN,NaN,NaN,NaN
232712,NaN,NaN,NaN,NaN,NaN,NaN,Imputed,NaN,NaN,NaN,NaN,NaN


In [17]:
#Here, we ensure there are no imputations if we already have estimates from data for an ISO3/disease/year/animal combination 
duplicated_rows = compiled.duplicated(subset=["Disease", "ISO3", "Year","Animal"], keep=False)

# Split the dataframe into duplicates and non-duplicates
duplicates = compiled[duplicated_rows]
non_duplicates = compiled[~duplicated_rows]

#Filter duplicates to only resolve those with non-missing "Incidence"
duplicates_with_incidence = duplicates[duplicates["Incidence (Cases per 100,000)"].notna()]

# Resolve duplicates by keeping the row with "Source" not equal to "Imputed"
resolved_duplicates = (
    duplicates_with_incidence.sort_values(by=["ISO3", "Animal","Disease", "Year", "Source"], 
                                          key=lambda col: col != "Imputed", 
                                          ascending=False)
    .drop_duplicates(subset=["Disease", "ISO3", "Year"], keep="first")
)

# Include duplicates without "Incidence" as they are
remaining_duplicates = duplicates[duplicates["Incidence (Cases per 100,000)"].isna()]

# Combine resolved duplicates, remaining duplicates, and non-duplicates
final_df = pd.concat([non_duplicates, resolved_duplicates, remaining_duplicates])


In [18]:
#Here, if we had an imputation of incidence due to unavailable population size (but we had reported cases by the country),
    #Then we use both datapoints and update the "Source" of the datapoint to point to both sources (the Data, and Imputed)

# We identify duplicates based on "ISO3", "Animal", "Year", "Disease"
duplicate_mask = final_df.duplicated(subset=["ISO3", "Animal", "Year", "Disease"], keep=False)

duplicates = final_df[duplicate_mask]
non_duplicates = final_df[~duplicate_mask]

# Processing duplicates to create combined records
processed_rows = []
for group_keys, group in duplicates.groupby(["ISO3", "Animal", "Year", "Disease"]):
    row_with_incidence = group[group["Incidence (Cases per 100,000)"].notna()].iloc[0]
    row_with_aggregate = group[group["Aggregate of Latest Reported Cases (up to Year)"].notna()].iloc[0]

    combined_row = row_with_aggregate.copy() 
    combined_row["Incidence (Cases per 100,000)"] = row_with_incidence["Incidence (Cases per 100,000)"]
    combined_row["Incidence (Cases per 100,000) Lower"] = row_with_incidence["Incidence (Cases per 100,000) Lower"]
    combined_row["Incidence (Cases per 100,000) Upper"] = row_with_incidence["Incidence (Cases per 100,000) Upper"]
    combined_row["Aggregate of Latest Reported Cases (up to Year)"] = row_with_aggregate["Aggregate of Latest Reported Cases (up to Year)"]
    combined_row["Source"] = f"Incidence Source: {row_with_incidence['Source']}, Aggregate Source: {row_with_aggregate['Source']}. Population size unavailable"

    processed_rows.append(combined_row)

processed_df = pd.DataFrame(processed_rows)

final_df = pd.concat([non_duplicates, processed_df], ignore_index=True)


In [19]:
# Just double checking year is correct for a few edge cases
mask = (final_df['Source'].str.contains('Imputed')) & (final_df['Source'] != 'Imputed')
adjusted_count = 0

def update_year_data(row):
    global adjusted_count  #
    if '-' in row['Year Data']:  
        parts = row['Year Data'].split('-') 
        if int(parts[-1])>row['Year']: 
            parts[-1] = str(int(row['Year']))  
            adjusted_count += 1  
        return '-'.join(parts) 
    return row['Year Data']  

final_df.loc[mask, 'Year Data'] = final_df.loc[mask].apply(update_year_data, axis=1)
print(f"Number of rows adjusted: {adjusted_count}")


Number of rows adjusted: 0


In [20]:
disease_animals_unique = pd.concat([final_df,vaccine_df])[['Disease','Animal']].drop_duplicates()


In [21]:
presence_absence_disease=pd.read_csv(os.path.join(source_data_path, 'Processed data','Multisource','domestic_disease_presence_or_absence_filter_and_susceptible_countries.csv'))

In [22]:
presence_absence_disease = presence_absence_disease.merge(
    pd.DataFrame({'Animal': ['Cattle', 'Pigs', 'Poultry']}),
    how='cross'
)
presence_absence_disease = presence_absence_disease.merge(
    pd.DataFrame(disease_animals_unique, columns=["Disease","Animal"]),
    on=["Disease","Animal"], how="inner"
)

In [23]:
#This code replaces incidence with 0 if disease is absent
k4 = ['ISO3','Animal','Disease','Year']
zero_cols = ['Incidence (Cases per 100,000)',
             'Incidence (Cases per 100,000) Lower',
             'Incidence (Cases per 100,000) Upper',
             'Aggregate of Latest Reported Cases (up to Year)',
             'Estimated (Scaled) Cases']

# from presence_absence_disease with Status=='Absent'
pad_no = (presence_absence_disease
          .loc[presence_absence_disease['Status in livestock'].eq('Absent'),
               k4 + ['Source','Year data']]
          .drop_duplicates(k4, keep='last'))

final_df = final_df.copy()

# indices for fast set/update
fidx = pd.MultiIndex.from_frame(final_df[k4])
nidx = pd.MultiIndex.from_frame(pad_no[k4])
look = pad_no.set_index(k4)

# update to 0 
m = fidx.isin(nidx)
if m.any():
    final_df.loc[m, zero_cols] = 0
    final_df.loc[m, 'Source']    = look.loc[fidx[m], 'Source'].to_numpy()
    final_df.loc[m, 'Year Data'] = look.loc[fidx[m], 'Year data'].to_numpy()

# insert missing with zeros-
miss_idx = nidx.difference(fidx)
if len(miss_idx):
    add = look.loc[miss_idx].reset_index().rename(columns={'Year data':'Year Data'})
    for c in final_df.columns:
        if c not in add.columns:
            add[c] = pd.NA
    add[zero_cols] = 0
    final_df = pd.concat([final_df, add[final_df.columns]], ignore_index=True)


In [24]:
#Code to add source for disease presence status when imputing incidence

k4 = ['ISO3','Year','Animal','Disease']

# Keeping source for rows where status == Present
pad_src = (presence_absence_disease
           .loc[presence_absence_disease['Status in livestock'].eq('Present'),
                k4 + ['Source']]
           .drop_duplicates(k4, keep='last')
           .rename(columns={'Source':'PAD_Source'}))

# Left-join to isolate source
final_df = final_df.merge(pad_src, on=k4, how='left')

mask = final_df['Source'].str.contains(r'\bImputed\b', na=False) & final_df['PAD_Source'].notna()

rep_vals = 'Imputed; ' + final_df.loc[mask, 'PAD_Source'].astype(str)
src_vals = final_df.loc[mask, 'Source']

final_df.loc[mask, 'Source'] = [
    s.replace('Imputed', r, 1) for s, r in zip(src_vals, rep_vals)
]

# 2) Correcting Source input
final_df['Source'] = final_df['Source'].str.replace(
    'Imputed; WAHIS administrative division report',
    'Imputed; WAHIS administrative division report (number of cases not provided)',
    n=1, regex=False
)


In [25]:
#Drop imputations when disease situation is unknown
k4 = ['ISO3','Year','Animal','Disease']

unknown_idx = pd.MultiIndex.from_frame(
    presence_absence_disease.loc[
        presence_absence_disease['Status in livestock'].eq('Unknown'),
        k4
    ].drop_duplicates()
)

mask_drop = pd.MultiIndex.from_frame(final_df[k4]).isin(unknown_idx)

# Keep only non-unknown rows
final_df = final_df.loc[~mask_drop].copy()


In [26]:
#Below code reconciles the categories 'bovine TB' and 'mycoplasma TB'
df   = final_df
KEYS = ['ISO3','Animal','Year']
VALS = ['Incidence (Cases per 100,000)',
        'Incidence (Cases per 100,000) Lower',
        'Incidence (Cases per 100,000) Upper',
        'Estimated (Scaled) Cases']

btb_mask  = df['Disease'].str.lower().eq('Bovine tuberculosis (-2018)')
myco_mask = df['Disease'].str.lower().eq('Mycobacterium tuberculosis complex (Inf. with)(2019-)')

# Keep original row indices to update in place
btb  = df.loc[btb_mask, KEYS + VALS + ['Source']].copy()
myco = df.loc[myco_mask, KEYS + VALS + ['Source']].copy()
btb['idx_btb']   = btb.index
myco['idx_myco'] = myco.index

btb  = btb.rename(columns={c: f'{c}_btb'  for c in VALS + ['Source']})
myco = myco.rename(columns={c: f'{c}_myco' for c in VALS + ['Source']})

pairs = btb.merge(myco, on=KEYS, how='inner')
pairs['_yr'] = pd.to_numeric(pairs['Year'], errors='coerce')

m1 = (pairs['_yr'] >= 2019) & (
     pairs['Incidence (Cases per 100,000)_btb'] > pairs['Incidence (Cases per 100,000)_myco'])

if m1.any():
    idx = pairs.loc[m1, 'idx_btb']
    df.loc[idx, VALS] = pairs.loc[m1, [f'{v}_myco' for v in VALS]].to_numpy()
    note = 'estimate clipped based on mycoplasma tuberculosis result'
    s = df.loc[idx, 'Source'].astype('string').fillna('')
    df.loc[idx, 'Source'] = np.where(s.eq(''), note, s + '; ' + note)

m2 = pairs['_yr'].between(2005, 2018) & (
     pairs['Incidence (Cases per 100,000)_myco'] < pairs['Incidence (Cases per 100,000)_btb'])

if m2.any():
    idx = pairs.loc[m2, 'idx_myco']
    df.loc[idx, VALS] = pairs.loc[m2, [f'{v}_btb' for v in VALS]].to_numpy()
    note = "estimate clipped based on 'bovine tuberculosis (-2018) result'"
    s = df.loc[idx, 'Source'].astype('string').fillna('')
    df.loc[idx, 'Source'] = np.where(s.eq(''), note, s + '; ' + note)


In [27]:
#Below code reconciles the categories 'low path avian influenza' and 'human-transmissibile low path avian influenza'
df   = final_df
KEYS = ['ISO3','Animal','Year']
VALS = ['Incidence (Cases per 100,000)',
        'Incidence (Cases per 100,000) Lower',
        'Incidence (Cases per 100,000) Upper',
        'Estimated (Scaled) Cases']

broad_inf_mask  = df['Disease'].str.lower().eq('Low pathogenic avian influenza (poultry) (2006-2021)')
human_t_inf_mask = df['Disease'].str.lower().eq('Low pathogenicity avian influenza viruses transmissible to humans (Inf. with) (2022-)')

# Keep original row indices to update in place
broad_inf  = df.loc[broad_inf_mask, KEYS + VALS + ['Source']].copy()
human_t_inf = df.loc[human_t_inf_mask, KEYS + VALS + ['Source']].copy()
broad_inf['idx_broad_inf']   = broad_inf.index
human_t_inf['idx_human_t_inf'] = human_t_inf.index

broad_inf  = broad_inf.rename(columns={c: f'{c}_broad_inf'  for c in VALS + ['Source']})
human_t_inf = human_t_inf.rename(columns={c: f'{c}_human_t_inf' for c in VALS + ['Source']})

pairs = broad_inf.merge(human_t_inf, on=KEYS, how='inner')
pairs['_yr'] = pd.to_numeric(pairs['Year'], errors='coerce')

m1 = (pairs['_yr'] >= 2022) & (
     pairs['Incidence (Cases per 100,000)_broad_inf'] < pairs['Incidence (Cases per 100,000)_human_t_inf'])

if m1.any():
    idx = pairs.loc[m1, 'idx_broad_inf']
    df.loc[idx, VALS] = pairs.loc[m1, [f'{v}_human_t_inf' for v in VALS]].to_numpy()
    note = 'estimate clipped based on human_t_infplasma tuberculosis result'
    s = df.loc[idx, 'Source'].astype('string').fillna('')
    df.loc[idx, 'Source'] = np.where(s.eq(''), note, s + '; ' + note)

m2 = pairs['_yr'].between(2005, 2021) & (
     pairs['Incidence (Cases per 100,000)_human_t_inf'] > pairs['Incidence (Cases per 100,000)_broad_inf'])

if m2.any():
    idx = pairs.loc[m2, 'idx_human_t_inf']
    df.loc[idx, VALS] = pairs.loc[m2, [f'{v}_broad_inf' for v in VALS]].to_numpy()
    note = "estimate clipped based on 'bovine tuberculosis (-2018) result'"
    s = df.loc[idx, 'Source'].astype('string').fillna('')
    df.loc[idx, 'Source'] = np.where(s.eq(''), note, s + '; ' + note)

    

In [28]:
# Since multiple categories exist for influenza virus, we set the broadest existing category, 'Influenza A virus (Inf. with)' to be 
# equal to the sum of cases in all other influenza categories IF the combined result from the other cases exceeds what is available for the broad category
#If no broad row exists, INSERT one using the summed values only when the aggregated
# 'Incidence (Cases per 100,000) Upper' > 0.

IAV = 'Influenza A virus (Inf. with)'
OTHERS = [
    'High pathogenicity avian influenza viruses (poultry) (Inf. with)',
    'Influenza A viruses of high pathogenicity (Inf. with) (non-poultry including wild birds) (2017-)',
    'Low pathogenic avian influenza (poultry) (2006-2021)',
    'Low pathogenicity avian influenza viruses transmissible to humans (Inf. with) (2022-)'
]
KEYS = ['ISO3','Year','Animal']
NUMS = ['Incidence (Cases per 100,000)',
        'Incidence (Cases per 100,000) Lower',
        'Incidence (Cases per 100,000) Upper',
        'Aggregate of Latest Reported Cases (up to Year)',
        'Estimated (Scaled) Cases']

INC_COL   = 'Incidence (Cases per 100,000)'
UPPER_COL = 'Incidence (Cases per 100,000) Upper'

df = final_df.copy()

rel = df.loc[df['Disease'].isin([IAV] + OTHERS)].copy()
others = rel.loc[rel['Disease'].isin(OTHERS)].copy()

for c in NUMS:
    others[c] = pd.to_numeric(others[c], errors='coerce').fillna(0)

def year_range_str(series):
    yrs = []
    for s in series.astype(str):
        yrs += [int(x) for x in re.findall(r'\d{4}', s)]
    if not yrs:
        return pd.NA
    mn, mx = min(yrs), max(yrs)
    return str(mn) if mn == mx else f'{mn}-{mx}'

agg_sum  = others.groupby(KEYS, as_index=False)[NUMS].sum()
agg_year = others.groupby(KEYS, as_index=False)['Year Data'].apply(year_range_str)\
                 .rename(columns={'Year Data':'Year Data agg'})
agg_cty  = others.groupby(KEYS, as_index=False)['Country'].first()\
                 .rename(columns={'Country':'Country first'})
agg = agg_sum.merge(agg_year, on=KEYS, how='left').merge(agg_cty, on=KEYS, how='left')

# existing IAV rows (first per key)
iav = rel.loc[rel['Disease'].eq(IAV)].sort_values(KEYS).drop_duplicates(KEYS, keep='first').copy()
iav['idx'] = iav.index  # original df indices

# Update when aggregated Incidence > current IAV Incidence
upd = iav.merge(agg, on=KEYS, how='inner', suffixes=('_iav','_agg'))
if len(upd):
    idx = upd['idx'].to_numpy()

    inc_iav = pd.to_numeric(upd[f'{INC_COL}_iav'], errors='coerce').fillna(0).to_numpy()
    inc_agg = pd.to_numeric(upd[f'{INC_COL}_agg'], errors='coerce').fillna(0).to_numpy()
    m_raise = (inc_agg > inc_iav)  # only update those keys where sum(OTHERS) exceeds current IAV

    if m_raise.any():
        idx_keep = idx[m_raise]
        agg_vals = upd[[f'{c}_agg' for c in NUMS]].apply(pd.to_numeric, errors='coerce').fillna(0).to_numpy()
        df.loc[idx_keep, NUMS] = agg_vals[m_raise]

        note = 'Reconciled: updated to sum of influenza subcategories (replaced lower total)'
        s = df.loc[idx_keep, 'Source'].astype('string').fillna('')
        mask_absent = s.str.contains('absence', case=False, na=False)

        df.loc[idx_keep, 'Source'] = np.where(
            mask_absent,
            note.lstrip('; '),  # overwrite when "absent" is present
            np.where(
                s.eq('') | s.isna(),
                note.lstrip('; '),                        # was empty → just note
                s.str.rstrip('; ').astype(str) + '; ' + note.lstrip('; ')  # append with "; "
            )
        )
        df.loc[idx_keep, 'Year Data'] = upd.loc[m_raise, 'Year Data agg'].to_numpy()

iav_keys = set(map(tuple, iav[KEYS].to_numpy()))
missing  = agg.loc[~agg[KEYS].apply(tuple, axis=1).isin(iav_keys)].copy()

# Inserts if missing IAV, and there are nonzero cases
if len(missing):
    missing = missing[pd.to_numeric(missing[UPPER_COL], errors='coerce').fillna(0) > 0]

    if len(missing):
        new = pd.DataFrame({
            'ISO3':    missing['ISO3'].to_numpy(),
            'Year':    missing['Year'].to_numpy(),
            'Animal':  missing['Animal'].to_numpy(),
            'Country': missing['Country first'].to_numpy(),
            'Disease': IAV,
            'Source':  'Reconciled: sum of reported cases from influenza subcategories (missing broad category report)',
            'Year Data': missing['Year Data agg'].to_numpy(),
            **{c: pd.to_numeric(missing[c], errors='coerce').fillna(0).to_numpy() for c in NUMS},
        })

        for c in df.columns:
            if c not in new.columns:
                new[c] = pd.NA
        new = new[df.columns]

        df = pd.concat([df, new], ignore_index=True)

final_df = df


In [29]:
# Same logic for influenza A virus, repeated for broad "Echinococcosis/hydatidosis" category, and "Echinococcus granulosus (Inf. with) (2014-)"/"Echinococcus multilocularis (Inf. with) (2014-)" contituents

IAV = 'Echinococcosis/hydatidosis'
OTHERS = [
    'Echinococcus granulosus (Inf. with) (2014-)',
    'Echinococcus multilocularis (Inf. with) (2014-)',
]
KEYS = ['ISO3','Year','Animal']
NUMS = ['Incidence (Cases per 100,000)',
        'Incidence (Cases per 100,000) Lower',
        'Incidence (Cases per 100,000) Upper',
        'Aggregate of Latest Reported Cases (up to Year)',
        'Estimated (Scaled) Cases']

INC_COL   = 'Incidence (Cases per 100,000)'
UPPER_COL = 'Incidence (Cases per 100,000) Upper'

df = final_df.copy()

rel = df.loc[df['Disease'].isin([IAV] + OTHERS)].copy()
others = rel.loc[rel['Disease'].isin(OTHERS)].copy()

for c in NUMS:
    others[c] = pd.to_numeric(others[c], errors='coerce').fillna(0)

def year_range_str(series):
    yrs = []
    for s in series.astype(str):
        yrs += [int(x) for x in re.findall(r'\d{4}', s)]
    if not yrs:
        return pd.NA
    mn, mx = min(yrs), max(yrs)
    return str(mn) if mn == mx else f'{mn}-{mx}'

agg_sum  = others.groupby(KEYS, as_index=False)[NUMS].sum()
agg_year = others.groupby(KEYS, as_index=False)['Year Data'].apply(year_range_str)\
                 .rename(columns={'Year Data':'Year Data agg'})
agg_cty  = others.groupby(KEYS, as_index=False)['Country'].first()\
                 .rename(columns={'Country':'Country first'})
agg = agg_sum.merge(agg_year, on=KEYS, how='left').merge(agg_cty, on=KEYS, how='left')

iav = rel.loc[rel['Disease'].eq(IAV)].sort_values(KEYS).drop_duplicates(KEYS, keep='first').copy()
iav['idx'] = iav.index  # original df indices

upd = iav.merge(agg, on=KEYS, how='inner', suffixes=('_iav','_agg'))
if len(upd):
    idx = upd['idx'].to_numpy()

    inc_iav = pd.to_numeric(upd[f'{INC_COL}_iav'], errors='coerce').fillna(0).to_numpy()
    inc_agg = pd.to_numeric(upd[f'{INC_COL}_agg'], errors='coerce').fillna(0).to_numpy()
    m_raise = (inc_agg > inc_iav)  # only update those keys where sum(OTHERS) exceeds current IAV

    if m_raise.any():
        idx_keep = idx[m_raise]
        # use the aggregated sums directly (replace, not add)
        agg_vals = upd[[f'{c}_agg' for c in NUMS]].apply(pd.to_numeric, errors='coerce').fillna(0).to_numpy()
        df.loc[idx_keep, NUMS] = agg_vals[m_raise]

        note = "; Reconciled: updated to sum of Eccinococcus spp. categories (adjusted broader category's lower total)"
        s = df.loc[idx_keep, 'Source'].astype('string').fillna('')
        mask_absent = s.str.contains('absence', case=False, na=False)

        df.loc[idx_keep, 'Source'] = np.where(
            mask_absent,
            note.lstrip('; '),  # overwrite when "absent" is present
            np.where(
                s.eq('') | s.isna(),
                note.lstrip('; '),                        # was empty → just note
                s.str.rstrip('; ').astype(str) + '; ' + note.lstrip('; ')  # append with "; "
            )
        )
        
        df.loc[idx_keep, 'Year Data'] = upd.loc[m_raise, 'Year Data agg'].to_numpy()

iav_keys = set(map(tuple, iav[KEYS].to_numpy()))
missing  = agg.loc[~agg[KEYS].apply(tuple, axis=1).isin(iav_keys)].copy()

if len(missing):
    missing = missing[pd.to_numeric(missing[UPPER_COL], errors='coerce').fillna(0) > 0]

    if len(missing):
        new = pd.DataFrame({
            'ISO3':    missing['ISO3'].to_numpy(),
            'Year':    missing['Year'].to_numpy(),
            'Animal':  missing['Animal'].to_numpy(),
            'Country': missing['Country first'].to_numpy(),
            'Disease': IAV,
            'Source':  'Reconciled: sum of reported cases from Eccinococcus spp. categories (broad category last reported absent, but species reported present)',
            'Year Data': missing['Year Data agg'].to_numpy(),
            **{c: pd.to_numeric(missing[c], errors='coerce').fillna(0).to_numpy() for c in NUMS},
        })

        for c in df.columns:
            if c not in new.columns:
                new[c] = pd.NA
        new = new[df.columns]

        df = pd.concat([df, new], ignore_index=True)

final_df = df


In [30]:
final_df['Country']=[flipped_dict[iso3] for iso3 in final_df['ISO3']]

In [31]:
final_df=final_df.sort_values(['Country','Animal','Year','Disease'])

In [32]:
#Removing imputations for countries that do not report to WAHIS, but had imputations informed via geographic location, animal population demographics, GDP etc.
countries_dont_report=['Virgin Islands, British',
'Antigua and Barbuda',
'Kosovo',
'Saint Kitts and Nevis',
'Gibraltar',
'Curaçao',
'Macao',
'Virgin Islands, U.S.',
'Saint Martin (French part)',
'Sint Maarten (Dutch part)',
'Tuvalu',
'Guam',
'Isle of Man',
'Monaco',
'Northern Mariana Islands',
'Niue',
'Turks and Caicos Islands',
'Tokelau',
'Bahamas',
'Puerto Rico',
'American Samoa',
'Marshall Islands',
'Nauru',
'Dominica',
'Bermuda']

ISO3s_no_reports=[countries[i] for i in countries_dont_report]

In [34]:
#Final formatting touches
final_df.drop(columns=['PAD_Source'],inplace=True)
final_df['Year Data']=[i.replace('.0','') if type(i)==str else i for i in final_df['Year Data'] ]

In [35]:
def load_valid_diseases(path):
    with Path(path).open("r", encoding="utf-8") as f:
        payload = json.load(f)

    data = payload["data"]
    out: Dict[str, Set[str]] = {}
    for animal, diseases in data.items():
        if not isinstance(diseases, list) or not all(isinstance(x, str) for x in diseases):
            raise ValueError("Each value must be a list[str]")
        out[str(animal)] = set(diseases)
    return out

valid_diseases = load_valid_diseases(os.path.join(source_data_path,'Processed data','WAHIS data','disease_animal_sets.json'))

In [36]:
# Building filter here to assure proper allocation of disease-free status, etc., to animals that can incur the disease ('disease-free' does not matter for animals who cannot be infected by the disease)
valid_pairs = {(animal, d) for animal, ds in valid_diseases.items() for d in ds}

pair_ok = pd.Series(list(zip(final_df["Animal"], 
                             final_df["Disease"])),
                    index=final_df.index).isin(valid_pairs)

# flag these with caution for next step, if Source mentions any of the keywords
has_flag = final_df["Source"].astype(str)\
    .str.contains(r"inferred|imputed|absence|disease-free|influenza", case=False, na=False)

# Safety check to removing allocations for diseases in animals that do not usually incur the disease (disease-free status is listed by country, not by animal, so this check is necessary)
to_remove = (~pair_ok) & has_flag

#Filter
final_df = final_df[~to_remove].copy()


In [37]:
#Adjusting source for diseases reconciled with others (shifting categories via WAHIS reporting, so we reconciled with best match to inform predictions)

mask = (
    (final_df['Disease'] == 'Mycobacterium tuberculosis complex (Inf. with)(2019-)') &
    (final_df['Year'] < 2019) &
    (~final_df['Source'].str.contains('bovine tuberculosis', case=False, na=False))
)

final_df.loc[mask, 'Source'] = (
    final_df.loc[mask, 'Source'].astype('string')
    + ' (Reconciled with Bovine tuberculosis (-2018))'
)

mask = (
    (final_df['Disease'] == 'Bovine tuberculosis (-2018)') &
    (final_df['Year'] >= 2019) &
    (~final_df['Source'].str.contains('Mycobacterium tuberculosis', case=False, na=False))
)

final_df.loc[mask, 'Source'] = (
    final_df.loc[mask, 'Source'].astype('string')
    + ' (Reconciled with Mycobacterium tuberculosis complex (Inf. with)(2019-))'
)
    
mask = (
    (final_df['Disease'] == 'Low pathogenic avian influenza (poultry) (2006-2021)') &
    (final_df['Year'] >= 2022) &
    (~final_df['Source'].str.contains('Low pathogenicity avian influenza', case=False, na=False))
)

final_df.loc[mask, 'Source'] = (
    final_df.loc[mask, 'Source'].astype('string')
    + ' (Reconciled with Low pathogenicity avian influenza viruses transmissible to humans (Inf. with) (2022-))'
)
    
    
mask = (
    (final_df['Disease'] == 'Low pathogenicity avian influenza viruses transmissible to humans (Inf. with) (2022-)') &
    (final_df['Year'] < 2022) &
    (~final_df['Source'].str.contains('Low pathogenic avian influenza', case=False, na=False))
)

final_df.loc[mask, 'Source'] = (
    final_df.loc[mask, 'Source'].astype('string')
    + ' (Reconciled with Low pathogenic avian influenza (poultry) (2006-2021))'
)

In [38]:
final_df[~final_df['ISO3'].isin(ISO3s_no_reports)].to_csv('Supplementary Spreadsheet- Incidence Estimates.csv',index=False,
                                                                      encoding='utf-8-sig'  # Use 'utf-8-sig' to handle special characters correctly
)